In [5]:
#导入库
import pandas as pd
import numpy as np
import sqlite3
import os
import glob
#设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option("display.width",1000)
#设置文件路径

data_path = './data/'
#读取所有数据
csv_files = glob.glob(data_path + '*.csv')
#建立字典
dataframes = {}
#遍历数据
for file in csv_files:
    name = os.path.basename(file).replace('.csv', '')  #替换将后缀替换为空
    dataframes[name] = pd.read_csv(file)


In [6]:
#查看订单表的数据类型
orders = dataframes["olist_orders_dataset"]
print(orders.dtypes)

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [7]:
#转换日期格式
data_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in data_columns:
    orders[col] = pd.to_datetime(orders[col])#依次取出在in后名称的列名，存到变量col，再将order表中的col转换为时间格式

print(orders.dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [14]:
#计算订单配送时长

orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders[[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_days"
]].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2.873877
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,2017-07-26 10:57:55,16.542245
6,136cce7faa42fdb2cefd53fdc79a6098,2017-04-11 12:22:08,NaT,NaN
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-16 13:10:30,2017-05-26 12:55:51,9.989826
8,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09,2017-02-02 14:08:10,9.818762
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29 11:55:02,2017-08-16 17:14:30,18.221852


In [15]:
#统计配送时长
print("平均配送时长", orders["delivery_days"].mean())
print("最长配送时长", orders["delivery_days"].max())
print("最短配送时长", orders["delivery_days"].min())
#配送时长最长有209天，存在异常

平均配送时长 12.558702304032053
最长配送时长 209.6286111111111
最短配送时长 0.5334143518518518


In [16]:
#找出最长的10笔订单
slow_orders = orders.sort_values(
    by = "delivery_days",
    ascending = False
)

slow_orders[[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_days"
]].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,209.628611
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,208.351759
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,195.634016
70307,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08 18:09:02,2017-09-19 14:33:17,194.850174
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,194.633611
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,194.049583
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,191.463542
81401,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,189.863160
54480,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,188.134618
68769,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,187.743843


In [18]:
#判断这些订单是否超过了预算日期。
orders["delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400#创建一个delay——day

slow_orders = orders.sort_values(
    by = "delivery_days",
    ascending = False
)#慢订单的定义

slow_orders[[
    "order_id",
    "delivery_days",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "delay_days"
]].head(10)

,order_id,delivery_days,order_estimated_delivery_date,order_delivered_customer_date,delay_days
19590,ca07593549f1816d26a572e06dc1eab6,209.628611,2017-03-22,2017-09-19 14:36:39,181.608785
55619,1b3190b2dfa9d789e1f14c05b647a14a,208.351759,2018-03-15,2018-09-19 23:24:07,188.975081
61610,440d0d17af552815d15a9e41abe49359,195.634016,2017-04-07,2017-09-19 15:12:50,165.633912
70307,2fb597c2f772eca01b1f5c561bf6cc7b,194.850174,2017-04-17,2017-09-19 14:33:17,155.606447
89130,285ab9426d6982034523a855f55a885e,194.633611,2017-04-06,2017-09-19 14:00:04,166.583380
38509,0f4519c5f1c541ddec9f21b3bddd533a,194.049583,2017-04-11,2017-09-19 14:38:21,161.609965
11399,47b40429ed8cce3aee9199792275433f,191.463542,2018-01-19,2018-07-13 20:51:31,175.869109
81401,2fe324febf907e3ea3f2aa9650869fa5,189.863160,2017-04-05,2017-09-19 17:00:07,167.708414
54480,2d7561026d542c8dbd8f0daeadf67a43,188.134618,2017-04-13,2017-09-19 14:38:18,159.609931
68769,c27815f7e3dd0b926b58552628481575,187.743843,2017-04-10,2017-09-19 17:14:25,162.718345


In [27]:
#整体延迟情况
print("延迟订单数量",(orders["delay_days"]>0).sum())

print("提前订单数量",(orders["delay_days"]<=0).sum())

print("平均延时天数",orders.loc[orders["delay_days"]>0, "delay_days"].mean())


延迟订单数量 7827
提前订单数量 88649
平均延时天数 9.552206167575203
